# ForecastMonitor: Tracking Performance and Alerting

In production forecasting, model accuracy degrades over time due to structural breaks,
regime changes, or data drift. A monitoring system that tracks performance and triggers
alerts is essential for maintaining forecast quality.

**ForecastMonitor** provides:
- Rolling and cumulative accuracy tracking (RMSE, MAE, MAPE)
- Bias evolution monitoring
- Forecast vs actual comparison with prediction intervals
- Degradation detection (recent vs historical accuracy)

**AlertSystem** adds rule-based alerting:
- Preset rules: RMSE spikes, bias drift, coverage drops, model changes
- Custom rules with configurable thresholds, windows, and severity levels
- Alert history and summary reporting

This notebook demonstrates the full monitoring lifecycle: from setting up a monitor,
tracking forecasts over time, detecting degradation, configuring alerts, and implementing
automated re-training responses.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.pipeline import (
    ForecastPipeline,
    ForecastMonitor,
    RecurringForecast,
    AlertSystem,
    AlertRule,
    Alert,
)

# Add helpers path
sys.path.insert(0, "../utils")
from helpers import load_macro_brazil

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("forecastbox monitoring modules loaded successfully.")

## 1. Setting Up the Monitor

The `ForecastMonitor` is attached to a `ForecastPipeline` and tracks pairs of
forecasted vs actual values over time. We set it up by:

1. Creating a pipeline for our target variable
2. Instantiating a `ForecastMonitor` linked to the pipeline
3. Populating it with historical forecast-actual pairs

In [ ]:
# Load macro_brazil dataset
df = load_macro_brazil()
print(f"Dataset: {df.shape[0]} months, columns: {list(df.columns)}")

# Create a pipeline for inflation forecasting
pipeline = ForecastPipeline(
    data_source=df,
    target="inflation",
    models=["auto_arima", "auto_ets", "naive"],
    combination="mean",
    evaluation=["rmse", "mae"],
    horizon=12,
    preprocess=["missing_fill"],
)

# Set up the monitor
monitor = ForecastMonitor(pipeline=pipeline)
print(f"\nMonitor created: {monitor}")
print(f"Actuals tracked: {len(monitor.actuals)}")
print(f"Forecasts tracked: {len(monitor.forecasted)}")

## 2. Tracking Forecast Accuracy Over Time

We simulate 12 months of one-step-ahead forecasting. At each month:
1. Use data up to month $t$ to fit models and forecast month $t+1$
2. When month $t+1$ is realized, record both forecast and actual
3. Track rolling accuracy metrics over time

In [ ]:
# Simulate 12 months of forecasts with progressive data arrival
# Hold out last 24 months, use first 12 for stable period, last 12 for degradation
n_holdout = 24
inflation = df["inflation"]
rng = np.random.default_rng(42)

for i in range(n_holdout):
    train_end = len(inflation) - n_holdout + i
    train_data = inflation.iloc[:train_end]
    
    # Create a simple pipeline for this month
    month_pipeline = ForecastPipeline(
        data_source=train_data.to_frame(),
        target="inflation",
        models=["auto_arima"],
        horizon=1,
        preprocess=["missing_fill"],
    )
    result = month_pipeline.run()
    
    # Get the 1-step-ahead forecast
    fc = list(result.forecasts.values())[0]
    forecast_date = inflation.index[train_end]
    actual_value = float(inflation.iloc[train_end])
    forecast_point = float(fc.point[0])
    
    # For the last 6 months, inject degradation (simulating structural break)
    if i >= 18:
        forecast_point += rng.normal(0.3, 0.15)  # systematic bias + noise
    
    lower_95 = forecast_point - 1.96 * float(fc.point.std()) if fc.lower_95 is not None else forecast_point - 0.3
    upper_95 = forecast_point + 1.96 * float(fc.point.std()) if fc.upper_95 is not None else forecast_point + 0.3
    
    # Record in monitor
    monitor.add_actual(forecast_date, actual_value)
    monitor.add_forecast(forecast_date, forecast_point, lower_95, upper_95)

print(f"Monitor now tracks {len(monitor.actuals)} actuals and {len(monitor.forecasted)} forecasts")

# Generate accuracy report
report = monitor.accuracy_report()
print(f"\n{report.summary()}")

## 3. Detecting Performance Degradation

Degradation detection compares recent forecast accuracy to historical accuracy.
If recent RMSE exceeds historical RMSE by more than a threshold (default: 1.5x),
degradation is flagged.

We also visualize rolling metrics over time to spot trends and structural breaks.

In [ ]:
# Test for degradation with different thresholds
for threshold in [1.2, 1.5, 2.0]:
    degraded = monitor.degradation_test(window=6, threshold=threshold)
    status = "DEGRADED" if degraded else "OK"
    print(f"Threshold {threshold}x: {status}")

# Rolling accuracy metrics
rolling_rmse = monitor.rolling_accuracy(window=6, metric="rmse")
rolling_mae = monitor.rolling_accuracy(window=6, metric="mae")
cumulative_rmse = monitor.cumulative_accuracy(metric="rmse")

# Bias tracker
bias = monitor.bias_tracker()

# Plot metrics evolution with degradation zone
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rolling RMSE
ax = axes[0, 0]
ax.plot(rolling_rmse.index, rolling_rmse.values, "b-", linewidth=2, label="Rolling RMSE (w=6)")
ax.plot(cumulative_rmse.index, cumulative_rmse.values, "b--", linewidth=1.2, label="Cumulative RMSE")
# Mark degradation zone (last 6 months)
if len(rolling_rmse) >= 6:
    ax.axvspan(rolling_rmse.index[-6], rolling_rmse.index[-1], alpha=0.15, color="red", label="Degradation zone")
ax.set_title("RMSE Evolution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Rolling MAE
ax = axes[0, 1]
ax.plot(rolling_mae.index, rolling_mae.values, "g-", linewidth=2, label="Rolling MAE (w=6)")
ax.set_title("MAE Evolution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bias evolution
ax = axes[1, 0]
ax.plot(bias.index, bias.values, "darkorange", linewidth=2, label="Cumulative Bias (MFE)")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Bias (Mean Forecast Error) Evolution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Forecast vs Actual
monitor.plot_forecast_vs_actual(ax=axes[1, 1])

plt.suptitle("Performance Degradation Detection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Alert System

The `AlertSystem` provides rule-based alerting on top of a `ForecastMonitor`.
Rules can be defined manually or loaded from presets:

| Preset | Metric | Condition | Threshold | Severity |
|--------|--------|-----------|-----------|----------|
| `rmse_spike` | RMSE | above | 1.5x historical | warning |
| `bias_drift` | Bias | above | 0.5 | warning |
| `coverage_drop` | Hit rate | below | 80% | critical |
| `model_change` | RMSE | change | 30% | info |

In [ ]:
# Create an AlertSystem linked to the monitor
alert_system = AlertSystem(monitor=monitor)

# Add preset rules
alert_system.add_preset("rmse_spike")
alert_system.add_preset("bias_drift")
alert_system.add_preset("coverage_drop")

# Add a custom rule: MAE exceeds 0.3
alert_system.add_rule(
    name="high_mae",
    metric="mae",
    condition="above",
    threshold=1.5,
    window=6,
    severity="warning",
)

print(alert_system.summary())

# Check for triggered alerts
alerts = alert_system.check()

print(f"\n{'=' * 50}")
print(f"TRIGGERED ALERTS: {len(alerts)}")
print(f"{'=' * 50}")

for alert in alerts:
    severity_icon = {"info": "INFO", "warning": "WARN", "critical": "CRIT"}
    print(f"\n  [{severity_icon.get(alert.severity, '?')}] {alert.rule}")
    print(f"    Metric value: {alert.metric_value:.4f}")
    print(f"    Threshold:    {alert.threshold}")
    print(f"    Message:      {alert.message}")

## 5. Model Comparison Dashboard

A visual dashboard summarizing model performance: rolling metrics, model ranking,
forecast vs actual comparison, and alert timeline.

In [ ]:
# --- Model Comparison Dashboard ---
# Simulate monitoring for multiple models to compare them

model_names = ["auto_arima", "auto_ets", "naive"]
model_monitors = {}

for model_name in model_names:
    m = ForecastMonitor(pipeline=pipeline)
    model_rng = np.random.default_rng(42 + hash(model_name) % 1000)
    
    for i in range(n_holdout):
        train_end = len(inflation) - n_holdout + i
        actual_val = float(inflation.iloc[train_end])
        forecast_date = inflation.index[train_end]
        
        # Simulate model-specific forecasts with different error profiles
        base_forecast = actual_val + model_rng.normal(0, 0.08)
        if model_name == "naive":
            base_forecast = float(inflation.iloc[train_end - 1])  # naive = last value
        elif model_name == "auto_ets":
            base_forecast = actual_val + model_rng.normal(0.02, 0.1)
        
        # Inject degradation in last 6 months for all models
        if i >= 18:
            base_forecast += model_rng.normal(0.15, 0.1)
        
        m.add_actual(forecast_date, actual_val)
        m.add_forecast(forecast_date, base_forecast,
                       base_forecast - 0.3, base_forecast + 0.3)
    
    model_monitors[model_name] = m

# Dashboard: 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Rolling RMSE comparison
ax = axes[0, 0]
for name, m in model_monitors.items():
    rolling = m.rolling_accuracy(window=6, metric="rmse")
    if not rolling.empty:
        ax.plot(rolling.index, rolling.values, linewidth=2, label=name)
ax.set_title("Rolling RMSE by Model (window=6)", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Model ranking (overall metrics)
ax = axes[0, 1]
ranking_data = []
for name, m in model_monitors.items():
    report = m.accuracy_report()
    ranking_data.append({
        "model": name,
        "RMSE": report.overall_metrics.get("rmse", np.nan),
        "MAE": report.overall_metrics.get("mae", np.nan),
        "Bias": abs(report.overall_metrics.get("mfe", 0)),
    })
ranking_df = pd.DataFrame(ranking_data).set_index("model")
ranking_df.plot(kind="bar", ax=ax, color=["steelblue", "darkorange", "forestgreen"])
ax.set_title("Overall Metrics by Model", fontsize=12, fontweight="bold")
ax.set_ylabel("Metric Value")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=0)

# Panel 3: Forecast vs Actual (best model)
best_model_name = ranking_df["RMSE"].idxmin()
model_monitors[best_model_name].plot_forecast_vs_actual(ax=axes[1, 0])
axes[1, 0].set_title(f"Forecast vs Actual ({best_model_name})", fontsize=12, fontweight="bold")

# Panel 4: Alert timeline
ax = axes[1, 1]
alert_history = alert_system.history()
if alert_history:
    severities = {"info": 0, "warning": 1, "critical": 2}
    colors_map = {"info": "steelblue", "warning": "orange", "critical": "red"}
    for alert in alert_history:
        ax.scatter(alert.timestamp, severities.get(alert.severity, 0),
                   color=colors_map.get(alert.severity, "gray"), s=150, zorder=5,
                   edgecolors="black", linewidth=0.5)
        ax.annotate(alert.rule, (alert.timestamp, severities.get(alert.severity, 0)),
                    textcoords="offset points", xytext=(5, 10), fontsize=8)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["Info", "Warning", "Critical"])
    ax.set_title("Alert Timeline", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No alerts triggered", ha="center", va="center",
            transform=ax.transAxes, fontsize=14)
    ax.set_title("Alert Timeline", fontsize=12, fontweight="bold")

plt.suptitle("Forecast Monitoring Dashboard", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Print ranking
print("\nModel Ranking:")
display(ranking_df.sort_values("RMSE").style.highlight_min(axis=0, color="lightgreen"))

## 6. Automated Response

When degradation is detected, we can automatically re-train the pipeline with
fresh data. This implements a simple automated response loop:

1. Check alerts
2. If critical/warning alerts are triggered, re-train the pipeline
3. Update the monitor with new forecasts
4. Verify improvement

In [ ]:
# Automated re-training loop
# Simulate: if degradation detected, re-train pipeline on latest data

def automated_retrain(
    pipeline: ForecastPipeline,
    monitor: ForecastMonitor,
    alert_system: AlertSystem,
    data: pd.DataFrame,
) -> dict:
    """Check alerts and retrain pipeline if degradation detected."""
    
    result = {
        "action_taken": False,
        "alerts_before": [],
        "degraded_before": False,
        "degraded_after": False,
    }
    
    # Step 1: Check for alerts
    alerts = alert_system.check()
    result["alerts_before"] = [a.rule for a in alerts]
    
    # Step 2: Check degradation
    degraded = monitor.degradation_test(window=6, threshold=1.5)
    result["degraded_before"] = degraded
    
    if not degraded and len(alerts) == 0:
        print("No degradation detected. No action needed.")
        return result
    
    print(f"Degradation detected! {len(alerts)} alert(s) triggered.")
    print("Re-training pipeline with latest data...\n")
    
    # Step 3: Re-train pipeline with full data
    pipeline.data_source = data
    new_results = pipeline.run()
    result["action_taken"] = True
    
    print(f"Re-trained pipeline:")
    print(f"  Best model: {new_results.best_model()}")
    print(f"  Models: {list(new_results.forecasts.keys())}")
    
    # Step 4: Generate new forecast and add to monitor
    if new_results.combination is not None:
        fc = new_results.combination
    else:
        fc = list(new_results.forecasts.values())[0]
    
    # Add the new forecast (first step ahead) to monitor
    if fc.index is not None and len(fc.index) > 0:
        monitor.add_forecast(
            date=fc.index[0],
            point=float(fc.point[0]),
            lower_95=float(fc.lower_95[0]) if fc.lower_95 is not None else None,
            upper_95=float(fc.upper_95[0]) if fc.upper_95 is not None else None,
        )
        print(f"  New forecast added for {fc.index[0]}: {fc.point[0]:.4f}")
    
    return result


# Run the automated response
print("=" * 60)
print("AUTOMATED RE-TRAINING RESPONSE")
print("=" * 60)
print()

# Check current state
report_before = monitor.accuracy_report()
print(f"Current state:")
print(f"  Overall RMSE: {report_before.overall_metrics.get('rmse', 'N/A'):.4f}")
print(f"  Overall MAE:  {report_before.overall_metrics.get('mae', 'N/A'):.4f}")
print(f"  Bias:         {report_before.bias:.4f}")
print(f"  Hit rate:     {report_before.hit_rate:.1%}")
print()

# Run automated retrain
retrain_result = automated_retrain(pipeline, monitor, alert_system, df)

print(f"\n{'=' * 60}")
print(f"RESULT:")
print(f"  Action taken:     {retrain_result['action_taken']}")
print(f"  Alerts triggered: {retrain_result['alerts_before']}")
print(f"  Degraded before:  {retrain_result['degraded_before']}")
print(f"{'=' * 60}")

# Show updated alert history
print(f"\nFull alert history ({len(alert_system.history())} alerts):")
for alert in alert_system.history():
    print(f"  [{alert.severity.upper():8s}] {alert.rule}: {alert.message}")

### Exercise 1: Set up monitoring for exchange rate forecast

Load `macro_brazil.csv` and create a `ForecastPipeline` for `exchange_rate` (BRL/USD).
Set up a `ForecastMonitor` with custom thresholds (MAE > 0.5, RMSE > 0.7).
Simulate 12 months of forecasts, plot metrics with alert zones.

In [ ]:
# Exercise 1 - Monitor for exchange rate with custom thresholds
#
# Strategy:
# - Exchange rate (BRL/USD) is more volatile than inflation
# - Custom thresholds: MAE > 0.5, RMSE > 0.7 (tighter than defaults)
# - Simulate 12 months of one-step-ahead forecasting
# - Inject degradation in last 4 months (regime change scenario)

# Step 1: Load data and create pipeline for exchange rate
df_ex1 = load_macro_brazil()
exchange_rate = df_ex1["exchange_rate"]

print(f"Exchange rate series: {len(exchange_rate)} observations")
print(f"Range: {exchange_rate.min():.2f} to {exchange_rate.max():.2f}")
print(f"Mean: {exchange_rate.mean():.4f}, Std: {exchange_rate.std():.4f}")

# Create pipeline for exchange rate
ex1_pipeline = ForecastPipeline(
    data_source=df_ex1,
    target="exchange_rate",
    models=["auto_arima", "auto_ets", "naive"],
    combination="mean",
    evaluation=["rmse", "mae"],
    horizon=12,
    preprocess=["missing_fill"],
)

# Step 2: Set up monitor
ex1_monitor = ForecastMonitor(pipeline=ex1_pipeline)

# Step 3: Simulate 12 months of one-step-ahead forecasts
n_sim = 12
rng_ex1 = np.random.default_rng(42)

print(f"\nSimulating {n_sim} months of exchange rate forecasts...")

for i in range(n_sim):
    train_end = len(exchange_rate) - n_sim + i
    train_data = exchange_rate.iloc[:train_end]
    
    # Fit pipeline on training data
    month_pipe = ForecastPipeline(
        data_source=train_data.to_frame(),
        target="exchange_rate",
        models=["auto_arima"],
        horizon=1,
        preprocess=["missing_fill"],
    )
    res = month_pipe.run()
    
    fc = list(res.forecasts.values())[0]
    forecast_date = exchange_rate.index[train_end]
    actual_value = float(exchange_rate.iloc[train_end])
    forecast_point = float(fc.point[0])
    
    # Inject degradation in last 4 months (simulating sudden BRL depreciation)
    if i >= 8:
        forecast_point += rng_ex1.normal(-0.4, 0.2)  # forecast misses depreciation
    
    lower_95 = forecast_point - 0.5
    upper_95 = forecast_point + 0.5
    
    ex1_monitor.add_actual(forecast_date, actual_value)
    ex1_monitor.add_forecast(forecast_date, forecast_point, lower_95, upper_95)
    
    error = abs(forecast_point - actual_value)
    print(f"  Month {i + 1:2d}: actual={actual_value:.3f}, "
          f"forecast={forecast_point:.3f}, |error|={error:.3f}"
          f"{'  ** DEGRADED' if i >= 8 else ''}")

# Step 4: Accuracy report
ex1_report = ex1_monitor.accuracy_report()
print(f"\n{'=' * 50}")
print("ACCURACY REPORT")
print(f"{'=' * 50}")
print(ex1_report.summary())

# Step 5: Set up alert system with custom thresholds
ex1_alerts = AlertSystem(monitor=ex1_monitor)

# Custom rules matching specification: MAE > 0.5, RMSE > 0.7
ex1_alerts.add_rule(
    name="mae_threshold",
    metric="mae",
    condition="above",
    threshold=0.5,
    window=6,
    severity="warning",
)
ex1_alerts.add_rule(
    name="rmse_threshold",
    metric="rmse",
    condition="above",
    threshold=0.7,
    window=6,
    severity="critical",
)

# Also add standard presets for reference
ex1_alerts.add_preset("bias_drift")
ex1_alerts.add_preset("coverage_drop")

# Check alerts
triggered = ex1_alerts.check()

print(f"\n{'=' * 50}")
print(f"ALERT CHECK: {len(triggered)} alert(s) triggered")
print(f"{'=' * 50}")
for alert in triggered:
    severity_icon = {"info": "INFO", "warning": "WARN", "critical": "CRIT"}
    print(f"  [{severity_icon.get(alert.severity, '?')}] {alert.rule}")
    print(f"    Value: {alert.metric_value:.4f}, Threshold: {alert.threshold}")
    print(f"    {alert.message}")

# Step 6: Plot metrics with alert thresholds
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Rolling RMSE with threshold line
ax = axes[0, 0]
rolling_rmse_ex1 = ex1_monitor.rolling_accuracy(window=4, metric="rmse")
if not rolling_rmse_ex1.empty:
    ax.plot(rolling_rmse_ex1.index, rolling_rmse_ex1.values, "b-", linewidth=2, label="Rolling RMSE (w=4)")
ax.axhline(0.7, color="red", linestyle="--", linewidth=1.5, label="Threshold (0.7)")
ax.set_title("Rolling RMSE vs Alert Threshold", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Rolling MAE with threshold line
ax = axes[0, 1]
rolling_mae_ex1 = ex1_monitor.rolling_accuracy(window=4, metric="mae")
if not rolling_mae_ex1.empty:
    ax.plot(rolling_mae_ex1.index, rolling_mae_ex1.values, "g-", linewidth=2, label="Rolling MAE (w=4)")
ax.axhline(0.5, color="red", linestyle="--", linewidth=1.5, label="Threshold (0.5)")
ax.set_title("Rolling MAE vs Alert Threshold", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bias tracker
ax = axes[1, 0]
bias_ex1 = ex1_monitor.bias_tracker()
if not bias_ex1.empty:
    ax.plot(bias_ex1.index, bias_ex1.values, "darkorange", linewidth=2, label="Cumulative Bias")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Bias Evolution (Exchange Rate)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Forecast vs Actual
ex1_monitor.plot_forecast_vs_actual(ax=axes[1, 1])
axes[1, 1].set_title("Forecast vs Actual (Exchange Rate)", fontsize=12, fontweight="bold")

plt.suptitle("Exercise 1: Exchange Rate Monitoring with Custom Thresholds",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Reference values
print("\n" + "-" * 40)
print("REFERENCE VALUES")
print("-" * 40)
print(f"Overall RMSE: {ex1_report.overall_metrics.get('rmse', np.nan):.4f}")
print(f"Overall MAE:  {ex1_report.overall_metrics.get('mae', np.nan):.4f}")
print(f"Bias (MFE):   {ex1_report.bias:.4f}")
print(f"Hit rate:     {ex1_report.hit_rate:.1%}")
print(f"Alerts triggered: {len(triggered)}")
print(f"Custom thresholds: MAE > 0.5 (warning), RMSE > 0.7 (critical)")

# Interpretation
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("""
Key findings:
1. Exchange rate forecasting is inherently harder than inflation forecasting
   due to higher volatility and sensitivity to external shocks (e.g., commodity
   prices, capital flows, geopolitical events).

2. The custom thresholds (MAE > 0.5, RMSE > 0.7) are calibrated for BRL/USD:
   - MAE > 0.5 means the model is off by more than R$0.50 on average
   - RMSE > 0.7 means large errors are dominating (squared penalty)
   These are reasonable alert levels for a currency that trades in the 3-6 range.

3. The injected depreciation shock (last 4 months) simulates a regime change
   that the model cannot anticipate. The monitoring system correctly detects
   this degradation through rising RMSE and MAE.

Recommendations:
- For exchange rates, consider shorter monitoring windows (3-4 months) since
  regime changes can be abrupt.
- Combine metric-based alerts with bias tracking: a rising bias indicates
  systematic under/over-prediction, not just noise.
- When alerts fire, investigate whether the degradation is due to a structural
  break (retrain) or temporary shock (wait for mean reversion).
""")

### Exercise 2: Custom alert rule — MASE > 1.5 for 3 consecutive months triggers retrain

Implement a custom monitoring rule: if MASE (Mean Absolute Scaled Error) exceeds 1.5
for 3 consecutive months, trigger a re-training of the pipeline. Simulate a scenario
where the model gradually degrades and the rule triggers a successful retrain.

In [ ]:
# Exercise 2 - Custom MASE > 1.5 for 3 consecutive months triggers retrain
#
# MASE (Mean Absolute Scaled Error) interpretation:
#   MASE = 1.0 -> model performs same as naive (random walk)
#   MASE < 1.0 -> model beats naive
#   MASE > 1.5 -> model is 50% worse than naive — serious degradation
#
# Strategy:
# - Simulate 18 months of forecasts with gradual degradation
# - Compute rolling MASE manually (ratio of MAE to naive MAE)
# - Track consecutive months where MASE > 1.5
# - When 3 consecutive months breach threshold, retrain
# - Verify improvement after retrain

# Step 1: Set up pipeline and monitor
df_ex2 = load_macro_brazil()
ex2_target = df_ex2["exchange_rate"]

ex2_pipeline = ForecastPipeline(
    data_source=df_ex2,
    target="exchange_rate",
    models=["auto_arima", "auto_ets"],
    combination="mean",
    evaluation=["rmse", "mae"],
    horizon=1,
    preprocess=["missing_fill"],
)

ex2_monitor = ForecastMonitor(pipeline=ex2_pipeline)

# Step 2: Set up AlertSystem with MAE-based rule
# (We use MAE with condition="above" as a proxy for MASE,
# and compute actual MASE separately for the consecutive-month logic)
ex2_alert_system = AlertSystem(monitor=ex2_monitor)
ex2_alert_system.add_rule(
    name="mase_degradation",
    metric="mae",
    condition="above",
    threshold=1.5,
    window=3,
    severity="critical",
)

# Step 3: Simulate 18 months with gradual degradation and MASE tracking
n_total = 18
rng_ex2 = np.random.default_rng(42)

# Storage for MASE computation
forecast_errors = []  # |forecast - actual|
naive_errors = []     # |actual_t - actual_{t-1}| (random walk benchmark)
monthly_mase = []     # MASE for each month
consecutive_breach = 0
retrain_triggered_at = None
retrained = False

print("Simulating 18 months with gradual degradation...")
print(f"{'Month':>5} {'Actual':>8} {'Forecast':>10} {'|Error|':>8} {'MASE':>8} {'Breach':>8} {'Action':>12}")
print("-" * 70)

for i in range(n_total):
    train_end = len(ex2_target) - n_total + i
    actual_value = float(ex2_target.iloc[train_end])
    forecast_date = ex2_target.index[train_end]
    
    # Naive forecast: last observed value
    naive_forecast = float(ex2_target.iloc[train_end - 1])
    naive_error = abs(actual_value - naive_forecast)
    naive_errors.append(naive_error)
    
    if not retrained:
        # Fit pipeline on training data
        train_data = ex2_target.iloc[:train_end]
        month_pipe = ForecastPipeline(
            data_source=train_data.to_frame(),
            target="exchange_rate",
            models=["auto_arima"],
            horizon=1,
            preprocess=["missing_fill"],
        )
        res = month_pipe.run()
        fc = list(res.forecasts.values())[0]
        forecast_point = float(fc.point[0])
        
        # Inject gradual degradation: bias increases linearly after month 8
        if i >= 8:
            degradation_factor = (i - 7) * 0.15  # grows: 0.15, 0.30, 0.45, ...
            forecast_point += rng_ex2.normal(degradation_factor, 0.1)
    else:
        # After retrain: use fresh pipeline (no degradation injection)
        train_data = ex2_target.iloc[:train_end]
        month_pipe = ForecastPipeline(
            data_source=train_data.to_frame(),
            target="exchange_rate",
            models=["auto_arima", "auto_ets"],
            combination="mean",
            horizon=1,
            preprocess=["missing_fill"],
        )
        res = month_pipe.run()
        if res.combination is not None:
            forecast_point = float(res.combination.point[0])
        else:
            fc = list(res.forecasts.values())[0]
            forecast_point = float(fc.point[0])
    
    # Record in monitor
    ex2_monitor.add_actual(forecast_date, actual_value)
    ex2_monitor.add_forecast(forecast_date, forecast_point,
                             forecast_point - 0.5, forecast_point + 0.5)
    
    # Compute forecast error
    fc_error = abs(forecast_point - actual_value)
    forecast_errors.append(fc_error)
    
    # Compute rolling MASE (over available history, min 1 month)
    if len(naive_errors) > 0 and np.mean(naive_errors) > 0:
        # Use rolling window of last 3 months for MASE
        window = min(3, len(forecast_errors))
        recent_mae = np.mean(forecast_errors[-window:])
        recent_naive_mae = np.mean(naive_errors[-window:])
        mase = recent_mae / max(recent_naive_mae, 1e-10)
    else:
        mase = np.nan
    monthly_mase.append(mase)
    
    # Check consecutive breach of MASE > 1.5
    action = ""
    if not np.isnan(mase) and mase > 1.5:
        consecutive_breach += 1
    else:
        consecutive_breach = 0
    
    # Trigger retrain if 3 consecutive months
    if consecutive_breach >= 3 and not retrained:
        retrain_triggered_at = i + 1
        retrained = True
        consecutive_breach = 0
        action = ">> RETRAIN"
    
    print(f"{i + 1:5d} {actual_value:8.3f} {forecast_point:10.3f} "
          f"{fc_error:8.3f} {mase:8.3f} {consecutive_breach:8d} {action:>12}")

# Step 4: Summary and analysis
print(f"\n{'=' * 60}")
print("MASE MONITORING SUMMARY")
print(f"{'=' * 60}")

# MASE evolution
mase_series = pd.Series(monthly_mase,
                        index=ex2_target.index[len(ex2_target) - n_total:
                                               len(ex2_target)])

print(f"\nRetrain triggered at month: {retrain_triggered_at}")
print(f"MASE before retrain (months 1-{retrain_triggered_at if retrain_triggered_at else n_total}): "
      f"{np.nanmean(monthly_mase[:retrain_triggered_at]):.4f}")
if retrain_triggered_at:
    print(f"MASE after retrain (months {retrain_triggered_at + 1}-{n_total}): "
          f"{np.nanmean(monthly_mase[retrain_triggered_at:]):.4f}")

# Accuracy report
ex2_report = ex2_monitor.accuracy_report()
print(f"\nOverall accuracy:")
print(f"  RMSE: {ex2_report.overall_metrics.get('rmse', np.nan):.4f}")
print(f"  MAE:  {ex2_report.overall_metrics.get('mae', np.nan):.4f}")
print(f"  Bias: {ex2_report.bias:.4f}")

# Check alerts
ex2_triggered = ex2_alert_system.check()
print(f"\nAlerts triggered: {len(ex2_triggered)}")
for alert in ex2_triggered:
    print(f"  [{alert.severity.upper()}] {alert.rule}: {alert.message}")

# Reference values
print(f"\n{'-' * 40}")
print("REFERENCE VALUES")
print(f"{'-' * 40}")
print(f"MASE threshold: 1.5")
print(f"Consecutive months required: 3")
print(f"Retrain triggered at month: {retrain_triggered_at}")
print(f"Total months simulated: {n_total}")
print(f"Degradation started at month: 9")

# Step 5: Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: MASE evolution with threshold and retrain marker
ax = axes[0, 0]
ax.plot(range(1, n_total + 1), monthly_mase, "b-o", linewidth=2, markersize=6, label="Rolling MASE (w=3)")
ax.axhline(1.5, color="red", linestyle="--", linewidth=1.5, label="Threshold (MASE=1.5)")
ax.axhline(1.0, color="gray", linestyle=":", linewidth=1, label="Naive benchmark (MASE=1.0)")
if retrain_triggered_at:
    ax.axvline(retrain_triggered_at, color="green", linestyle="-.", linewidth=2, label=f"Retrain (month {retrain_triggered_at})")
ax.set_xlabel("Month")
ax.set_ylabel("MASE")
ax.set_title("MASE Evolution with Retrain Trigger", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Forecast errors over time
ax = axes[0, 1]
ax.bar(range(1, n_total + 1), forecast_errors, color=["green" if e < 0.5 else "orange" if e < 1.0 else "red" for e in forecast_errors], alpha=0.7)
if retrain_triggered_at:
    ax.axvline(retrain_triggered_at, color="green", linestyle="-.", linewidth=2, label=f"Retrain")
ax.set_xlabel("Month")
ax.set_ylabel("|Forecast Error|")
ax.set_title("Absolute Forecast Errors", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")

# Panel 3: Forecast vs Actual
ex2_monitor.plot_forecast_vs_actual(ax=axes[1, 0])
if retrain_triggered_at:
    retrain_date = ex2_target.index[len(ex2_target) - n_total + retrain_triggered_at - 1]
    axes[1, 0].axvline(retrain_date, color="green", linestyle="-.", linewidth=2, label="Retrain")
    axes[1, 0].legend(fontsize=9)
axes[1, 0].set_title("Forecast vs Actual (Exchange Rate)", fontsize=12, fontweight="bold")

# Panel 4: Rolling accuracy comparison (before vs after retrain)
ax = axes[1, 1]
rolling_rmse_ex2 = ex2_monitor.rolling_accuracy(window=3, metric="rmse")
rolling_mae_ex2 = ex2_monitor.rolling_accuracy(window=3, metric="mae")
if not rolling_rmse_ex2.empty:
    ax.plot(rolling_rmse_ex2.index, rolling_rmse_ex2.values, "b-", linewidth=2, label="Rolling RMSE (w=3)")
if not rolling_mae_ex2.empty:
    ax.plot(rolling_mae_ex2.index, rolling_mae_ex2.values, "g-", linewidth=2, label="Rolling MAE (w=3)")
if retrain_triggered_at:
    ax.axvline(retrain_date, color="green", linestyle="-.", linewidth=2, label="Retrain")
ax.set_title("Rolling Accuracy Before/After Retrain", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle("Exercise 2: MASE-Based Retrain Rule (3 Consecutive Months > 1.5)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Interpretation
print("\n" + "=" * 60)
print("INTERPRETATION & RECOMMENDATIONS")
print("=" * 60)
print(f"""
Custom Rule: MASE > 1.5 for 3 Consecutive Months

What happened:
1. Months 1-8: Model performs well, MASE stays below 1.5 (better than or
   comparable to naive benchmark).
2. Month 9+: Gradual degradation injected (simulating structural break).
   MASE rises as forecast errors grow faster than naive errors.
3. Month {retrain_triggered_at}: After 3 consecutive months of MASE > 1.5, the retrain
   rule triggers. Pipeline is re-estimated with all available data.
4. Post-retrain: Model recovers, MASE drops back toward 1.0.

Why MASE > 1.5 for 3 months?
- MASE > 1.0 means worse than naive — a single month might be noise.
- MASE > 1.5 means 50% worse than naive — clearly degraded.
- 3 consecutive months filters out transient spikes and confirms
  persistent degradation before triggering an expensive retrain.

Practical recommendations:
1. MASE is preferred over raw MAE/RMSE for retrain rules because it is
   scale-independent and benchmarked against naive.
2. The consecutive-month requirement prevents false alarms from one-off
   shocks (e.g., a single outlier month).
3. After retrain, continue monitoring — if MASE stays high even after
   retrain, the model specification itself may need to change (add
   features, switch model class, etc.).
4. Consider a tiered response: MASE > 1.5 for 3 months = retrain;
   MASE > 2.0 for 1 month = immediate retrain (critical degradation).
""")